<a href="https://colab.research.google.com/github/eliasakalu/ICogLabs_AI_Projects/blob/main/RNN_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [231]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,SimpleRNN
#pip install python-docx  Uncomment it if you see the error for docx
from docx import Document
from google.colab import files

In [232]:
uploaded_file = files.upload()
uploaded_file = list(uploaded_file.keys())[0]

IndexError: list index out of range

In [ ]:
#Docx file handing
doc = Document(uploaded_file)
paragraph = "\n".join([para.text for para in doc.paragraphs if para.text.strip()!=""])

#txt file extension
# with open(uploaded_file,"r",encoding="utf-8") as file:
#   text = file.read()
# print(text)

In [ ]:
#The sample text we can take as dataset
text='''
This proposal outlines the development of a comprehensive Digital Marketplace and Supply Chain Platform specifically designed to address these challenges. Our solution will create a direct, transparent link between farmers and buyers, empowering farmers with real-time market information and fair value for their produce
'''
char = sorted(list(set(text)))
print(f"The Length of Alphabets from 52: {len(char)} was given.")

#Dictionary that matches Alphabet with Index
char_to_index = {char:i for i,char in enumerate(char)}
index_to_char = {i:char for i,char in enumerate(char)}
print(char)


In [233]:
seqLength = 5   #The numbers of letter the model learns at a time
sequences = []  #A place where those learned letter stored
labels = []     #The next letter comes after those 3 letters

for i in range(len(text)-seqLength):
  seq = text[i:i+seqLength]
  label = text[i+seqLength]

  sequences.append([char_to_index[char] for char in seq])
  labels.append([char_to_index[label]])
X = np.array(sequences)
Y = np.array(labels)
print(X.shape)

(317, 5)


In [234]:
#Encoding into One Hot Encoding
X_encoded = tf.one_hot(X,len(char))
Y_encoded = tf.one_hot(Y,len(char))
Y_encoded = np.array(Y_encoded).reshape((X.shape[0],len(char))) #Since it causes mismatch dimmension between X and Y during training
print(f"After encoded X shape{X_encoded.shape}")
print(f"After encoded and reshaping Y shape{Y_encoded.shape}")

After encoded X shape(317, 5, 34)
After encoded and reshaping Y shape(317, 34)


In [235]:
#Train our model
model = Sequential([
    SimpleRNN(128,input_shape=(seqLength,len(char)),activation="relu"),
    Dense(len(char),activation="softmax")
])

model.compile(optimizer="Adam",loss="categorical_crossentropy",metrics=["accuracy"])
model.summary()
model.fit(X_encoded,Y_encoded,epochs=100) #Trained using one hot encoding

Model: "sequential_29"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_29 (SimpleRNN)       │ (None, 128)            │        20,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 34)             │         4,386 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,250 (98.63 KB)

 Trainable params: 25,250 (98.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.0599 - loss: 3.4979    
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1293 - loss: 3.2957 
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1388 - loss: 3.0259 
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1546 - loss: 2.8967 
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1893 - loss: 2.8218 
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1924 - loss: 2.7416 
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2618 - loss: 2.6592 
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2839 - loss: 2.5587
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3218 - loss: 2.4599
Epoch 10/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3691 - loss: 2.3520
Epoch 11/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3817 - loss: 2.2327
Epoch 12/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/s

In [237]:
#Prediction maker
sample = "This proposal outlines "
generated = sample
for i in range(200):
  temp = generated[-seqLength:]  #Take the last 3 words eg: ils
  print(temp)
  x_temp = np.array([char_to_index[x] for x in temp])  #Convert it into array eg:[0,4,1]
  print(type(x_temp))
  X_temp_encoded = tf.one_hot(x_temp,len(char)) # change to 3row by 11 column
  X_temp_encoded = tf.reshape(X_temp_encoded,(1,seqLength,len(char))) #Changes the (3,11) to (1,3,11)with out affecting contents since it is compatible for model
  index = np.argmax(model.predict(X_temp_encoded))
  predicted_char = char[index]
  generated += predicted_char
print(generated)

ines 
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
nes t
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
es th
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
s the
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
 thes
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
these
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
hese 
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
ese c
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
se ch
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
e cha
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
 chal
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
chall
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
halle
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
allen
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
lleng
<class 'numpy.ndarray'>
1/1 ━━━━━━━━━━━